# Logistic Regression Deep Dive

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# 1. Load the Data
print("Loading Data...")
y_train = np.load('../data/vectors/y_train.npy', allow_pickle=True)
y_val = np.load('../data/vectors/y_val.npy', allow_pickle=True)
y_test = np.load('../data/vectors/y_test.npy', allow_pickle=True)

# Route A: TF-IDF + SVD
X_train_svd = np.load('../data/vectors/svd_train.npy')
X_val_svd = np.load('../data/vectors/svd_val.npy')
X_test_svd = np.load('../data/vectors/svd_test.npy')

# Route B: Dense Embeddings
X_train_embed = np.load('../data/vectors/embed_train.npy')
X_val_embed = np.load('../data/vectors/embed_val.npy')
X_test_embed = np.load('../data/vectors/embed_test.npy')

LABELS = ['agree', 'disagree', 'discuss', 'unrelated']

Loading Data...


In [3]:
# 2. Master Evaluation & Plotting Function
def evaluate_tuned_model(model, X_train, y_train, X_val, y_val, X_test, y_test, title_prefix=""):
    print(f"\n{'='*60}\n{title_prefix} MODEL EVALUATION\n{'='*60}")
    print(f"Best Parameters Found: {model.best_params_}")
    
    # Get Predictions
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    # Print Test Classification Report (What the Professor wants to see)
    print("\n--- HOLD-OUT TEST SET CLASSIFICATION REPORT ---")
    print(classification_report(y_test, y_pred_test, labels=LABELS, zero_division=0))
    
    # --- VISUALIZATIONS ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot 1: Train vs Val vs Test F1 Score Comparison (To check Overfitting)
    train_f1 = f1_score(y_train, y_pred_train, average=None, labels=LABELS)
    val_f1 = f1_score(y_val, y_pred_val, average=None, labels=LABELS)
    test_f1 = f1_score(y_test, y_pred_test, average=None, labels=LABELS)
    
    df_plot = pd.DataFrame({
        'Stance': LABELS * 3,
        'F1 Score': np.concatenate([train_f1, val_f1, test_f1]),
        'Split': ['Train']*4 + ['Validation']*4 + ['Test']*4
    })
    
    sns.barplot(data=df_plot, x='Stance', y='F1 Score', hue='Split', ax=axes[0], palette='viridis')
    axes[0].set_title(f"{title_prefix} - Overfit Check (F1 Scores)")
    axes[0].set_ylim(0, 1)
    
    # Plot 2: Test Set Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_test, labels=LABELS, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS, ax=axes[1])
    axes[1].set_title(f"{title_prefix} - Test Set Confusion Matrix")
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    
    plt.tight_layout()
    plt.show()
    
    # Return train/val/test macro F1 to quickly spot underfitting/overfitting
    macro_train = f1_score(y_train, y_pred_train, average='macro')
    macro_val = f1_score(y_val, y_pred_val, average='macro')
    macro_test = f1_score(y_test, y_pred_test, average='macro')
    
    print(f"Macro F1 -> Train: {macro_train:.4f} | Val: {macro_val:.4f} | Test: {macro_test:.4f}")
    if (macro_train - macro_val) > 0.15:
        print("⚠️ WARNING: Model is overfitting to the training data.")
    elif macro_train < 0.60:
        print("⚠️ WARNING: Model might be underfitting (high bias).")

In [4]:
# --- LOGISTIC REGRESSION SETUP ---
# We use 'saga' solver because it supports both L1 and L2 regularization
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', solver='saga', max_iter=2000, random_state=42))
])

# Define the Grid for Hyperparameter Tuning
# C is inverse of regularization strength. Smaller C = stronger regularization (prevents overfitting)
param_grid_lr = {
    'clf__penalty': ['l1', 'l2'],
    'clf__C': [0.01, 0.1, 1.0, 10.0]
}

print("Initiating GridSearchCV for Logistic Regression...")

# ==========================================
# ROUTE A: TF-IDF + SVD
# ==========================================
print("\n[1/2] Tuning Logistic Regression on Route A (TF-IDF + SVD)...")
grid_lr_svd = GridSearchCV(lr_pipeline, param_grid_lr, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_lr_svd.fit(X_train_svd, y_train)

# Evaluate
evaluate_tuned_model(grid_lr_svd, X_train_svd, y_train, X_val_svd, y_val, X_test_svd, y_test, title_prefix="LR (TF-IDF + SVD)")

# Save Model
joblib.dump(grid_lr_svd.best_estimator_, '../data/models/LR_best_tfidf_svd.pkl')

Initiating GridSearchCV for Logistic Regression...

[1/2] Tuning Logistic Regression on Route A (TF-IDF + SVD)...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


KeyboardInterrupt: 

In [ ]:
# ==========================================
# ROUTE B: DENSE EMBEDDINGS
# ==========================================
print("\n[2/2] Tuning Logistic Regression on Route B (Dense Embeddings)...")
grid_lr_embed = GridSearchCV(lr_pipeline, param_grid_lr, cv=3, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_lr_embed.fit(X_train_embed, y_train)

# Evaluate
evaluate_tuned_model(grid_lr_embed, X_train_embed, y_train, X_val_embed, y_val, X_test_embed, y_test, title_prefix="LR (Embeddings)")

# Save Model
import os
os.makedirs('../data/models', exist_ok=True)
joblib.dump(grid_lr_embed.best_estimator_, '../data/models/LR_best_embeddings.pkl')

print("\nSaved both tuned Logistic Regression models to '../data/models/'.")